&nbsp;

# ENCODE rE2G Result Visualization

- figshare に登録するデータの加工も行っていく

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt
from upsetplot import UpSet, from_contents
from IPython.display import display


os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

## Introduction

- なぜ ENCODE-rE2G を使用するに至ったのか
- -> Enhancer Level Activity (ELA) しか 近傍の CRE がない 転写産物が見られたことから
- きちんと数で示したいと考え、ここで処理

In [2]:
up5p_transcript_cre_pairs = pl.read_parquet(
    "../../out/fantabio/parquet_file/up5p_transcript_fantabio_v261001.parquet"
)

print(f"up5p_transcript - CRE pairs count: {up5p_transcript_cre_pairs.select(['transcript_id', 'cre_id']).n_unique()}")
print(f"up5p_transcript - CRE pairs count (only lncRNA): {up5p_transcript_cre_pairs.filter((pl.col('transcript_type') == 'lncRNA')).select(['transcript_id', 'cre_id']).n_unique()} transcript - CRE pairs")
print(f"up5p_transcripts (has fanta.bio CRE annotation): {up5p_transcript_cre_pairs.select('transcript_id').n_unique()} transcripts")
print(f"up5p lncRNA transcripts (has fanta.bio CRE annotation): {up5p_transcript_cre_pairs.filter(pl.col('transcript_type') == 'lncRNA').select('transcript_id').n_unique()} lncRNAs")
print(f"CREs (has fanta.bio transcript annotation): {up5p_transcript_cre_pairs.select('cre_id').n_unique()} CREs")
print(f"CREs (PLA class): {up5p_transcript_cre_pairs.filter(pl.col('fantabio_cre_class') == 'PLA').select('cre_id').n_unique()}")
print(f"CREs (ELA class): {up5p_transcript_cre_pairs.filter(pl.col('fantabio_cre_class') == 'ELA').select('cre_id').n_unique()}")

display(up5p_transcript_cre_pairs)

up5p_transcript - CRE pairs count: 63099
up5p_transcript - CRE pairs count (only lncRNA): 9221 transcript - CRE pairs
up5p_transcripts (has fanta.bio CRE annotation): 21696 transcripts
up5p lncRNA transcripts (has fanta.bio CRE annotation): 3657 lncRNAs
CREs (has fanta.bio transcript annotation): 33961 CREs
CREs (PLA class): 14620
CREs (ELA class): 19341


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red"""
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1,"""FCHS_28106""","""chr1""",161524161,"""161524546""","""cp4@HSPA6""","[""ENST00000309758.6"", ""NM_002155.5"", … ""AK295212""]","""0""",0,"[{""EH38E2843855"",""PLS""}]","[{""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX3322007"",560}]}, {""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX13551280"",511}]}, … {""MED12"",511,[{""SRX10852993"",511}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]","[{""hg19::chr1:161494328..161494332,+;hg_6264.1"",""p1@HSPA6""}]",null,0.943234,"""PLA""","""204,0,0""","""red"""
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1,"""FCHS_456076""","""chr1""",161524872,"""161524930""","""FCHS_456076""","[""KJ897015""]","""-214""",-332,"[{""EH38E2843856"",""pELS""}, {""EH38E3981534"",""PLS""}]","[{""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX12008350"",1465}]}, {""ZNF687"",2211,[{""SRX10478566"",2211}]}, … {""NR5A2"",516,[{""SRX10478059"",516}]}]","[{""HS_00013404"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00013405"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,1.0,"""ELA""","""255,202,74""","""yellow"""
"""ENST00000679223.2""",338,8,60,330,222,1,28,221,"""DNAJB1""","""HGNC:5270""","""protein_coding""","""ENSG00000132002.10""","[""alternative_5_UTR"", ""inferred_transcript_model"", … ""CCDS""]","""-""",5,null,null,"[""polya_site""]","""ensembl_havana_gene_homo_sapie…","[""alternative 5' UTR"", ""inferred transcri

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. Only ELA Count

- 念のため、distance のところもカウントしておく

In [3]:
up5p_transcript_cre_pair_select = up5p_transcript_cre_pairs.select(
    "transcript_id",
    "HN2",
    "transcript_type",
    "cre_id",
    "transcript_distance",
    "fantabio_cre_class"
).group_by(
    ["transcript_id", "HN2", "transcript_type"],
    maintain_order=True
).agg(
    pl.col("cre_id"),
    pl.col("fantabio_cre_class"),
    pl.col("transcript_distance")
).with_columns(
    # Count PLA / ELA in the CRE class list of each transcript
    pl.col("fantabio_cre_class").list.count_matches("PLA").alias("n_pla"),
    pl.col("fantabio_cre_class").list.count_matches("ELA").alias("n_ela"),
).with_columns(
    # True if all proximal CREs are ELA (at least one ELA and no PLA)
    ((pl.col("n_ela") > 0) & (pl.col("n_pla") == 0)).alias("is_only_ela"),
)

pl.Config.set_tbl_rows(4)
display(up5p_transcript_cre_pair_select)

transcript_id,HN2,transcript_type,cre_id,fantabio_cre_class,transcript_distance,n_pla,n_ela,is_only_ela
str,i64,str,list[str],list[str],list[i64],u32,u32,bool
"""ENST00000309758.6""",335,"""protein_coding""","[""FCHS_28105"", ""FCHS_28106"", ""FCHS_456076""]","[""PLA"", ""PLA"", ""ELA""]","[53, 0, -332]",2,1,false
"""ENST00000679223.2""",330,"""protein_coding""","[""FCHS_201778""]","[""PLA""]",[0],1,0,false
…,…,…,…,…,…,…,…,…
"""ENST00000480008.1""",44,"""retained_intron""","[""FCHS_373148"", ""FCHS_373149"", … ""FCHS_373152""]","[""ELA"", ""ELA"", … ""PLA""]","[-383, -162, … 286]",2,3,false
"""ENST00000774581.1""",44,"""lncRNA""","[""FCHS_400104""]","[""ELA""]",[294],0,1,true


In [4]:
up5p_transcript_cre_pair_select_lncRNA = up5p_transcript_cre_pair_select.filter(
    (pl.col("transcript_type") == "lncRNA")
    & (pl.col("is_only_ela") == True)
)

pl.Config.set_tbl_rows(5)
display(up5p_transcript_cre_pair_select_lncRNA)

transcript_id,HN2,transcript_type,cre_id,fantabio_cre_class,transcript_distance,n_pla,n_ela,is_only_ela
str,i64,str,list[str],list[str],list[i64],u32,u32,bool
"""ENST00000850140.1""",311,"""lncRNA""","[""FCHS_28129""]","[""ELA""]",[0],0,1,true
"""ENST00000778743.1""",289,"""lncRNA""","[""FCHS_180815"", ""FCHS_180816""]","[""ELA"", ""ELA""]","[0, -402]",0,2,true
"""ENST00000547824.1""",264,"""lncRNA""","[""FCHS_92253"", ""FCHS_92254""]","[""ELA"", ""ELA""]","[107, -187]",0,2,true
…,…,…,…,…,…,…,…,…
"""ENST00000812313.1""",44,"""lncRNA""","[""FCHS_40304"", ""FCHS_40305"", ""FCHS_460035""]","[""ELA"", ""ELA"", ""ELA""]","[0, -375, -375]",0,3,true
"""ENST00000774581.1""",44,"""lncRNA""","[""FCHS_400104""]","[""ELA""]",[294],0,1,true


In [5]:
up5p_transcript_cre_pair_only_ela = up5p_transcript_cre_pair_select.filter(
    pl.col("is_only_ela") == True
).with_columns(
    # True if any proximal ELA CRE overlaps the TSS (transcript_distance == 0)
    pl.col("transcript_distance").list.contains(0).alias("is_tss_overlap")
)

print(f"up5p transcripts with only ELA: {up5p_transcript_cre_pair_only_ela.select('transcript_id').n_unique()}")
print(f"  TSS overlaps an ELA CRE: {up5p_transcript_cre_pair_only_ela.filter(pl.col('is_tss_overlap')).select('transcript_id').n_unique()}")
print(f"  TSS does not overlap any CRE (ELA only nearby): {up5p_transcript_cre_pair_only_ela.filter(~pl.col('is_tss_overlap')).select('transcript_id').n_unique()}")

pl.Config.set_tbl_rows(10)
display(up5p_transcript_cre_pair_only_ela)

up5p transcripts with only ELA: 3921
  TSS overlaps an ELA CRE: 2304
  TSS does not overlap any CRE (ELA only nearby): 1617


transcript_id,HN2,transcript_type,cre_id,fantabio_cre_class,transcript_distance,n_pla,n_ela,is_only_ela,is_tss_overlap
str,i64,str,list[str],list[str],list[i64],u32,u32,bool,bool
"""ENST00000850140.1""",311,"""lncRNA""","[""FCHS_28129""]","[""ELA""]",[0],0,1,true,true
"""ENST00000333141.7""",295,"""protein_coding""","[""FCHS_309956""]","[""ELA""]",[0],0,1,true,true
"""ENST00000778743.1""",289,"""lncRNA""","[""FCHS_180815"", ""FCHS_180816""]","[""ELA"", ""ELA""]","[0, -402]",0,2,true,true
"""ENST00000450186.1""",286,"""protein_coding""","[""FCHS_60821"", ""FCHS_60822"", ""FCHS_60823""]","[""ELA"", ""ELA"", ""ELA""]","[423, 295, 0]",0,3,true,true
"""ENST00000671774.1""",282,"""protein_coding_CDS_not_defined""","[""FCHS_418486""]","[""ELA""]",[0],0,1,true,true
…,…,…,…,…,…,…,…,…,…
"""ENST00000704302.1""",44,"""nonsense_mediated_decay""","[""FCHS_493230"", ""FCHS_142626""]","[""ELA"", ""ELA""]","[0, 0]",0,2,true,true
"""ENST00000538456.4""",44,"""protein_coding""","[""FCHS_200368""]","[""ELA""]",[0],0,1,true,true
"""ENST00000596124.3""",44,"""protein_coding""","[""FCHS_513693""]","[""ELA""]",[0],0,1,true,true


In [6]:
up5p_transcript_cre_pair_only_ela_list = (
    up5p_transcript_cre_pair_only_ela
    .group_by("transcript_type")
    .agg(
        pl.col("transcript_id").n_unique().alias("n_transcript"),
        # Count only transcripts whose TSS overlaps an ELA CRE
        pl.col("transcript_id").filter(pl.col("is_tss_overlap")).n_unique().alias("n_tss_overlap"),
    )
    .sort("n_transcript", "transcript_type", descending=[True, False])
).rename(
    {
        "transcript_type": "Transcript Type",
        "n_transcript": "Only Proximal ELA Transcript Count",
        "n_tss_overlap": "TSS-overlapping ELA Transcript Count"
    }
)

# up5p_transcript_cre_pair_only_ela_list.write_excel(
#     "../../out/figshare/encode_re2g/up5p_lncRNA_only_proximal_ela_transcript.xlsx",
#     worksheet="up5p lncRNA only proxial ELA",
#     autofit=True
# )

pl.Config.set_tbl_rows(-1)
display(up5p_transcript_cre_pair_only_ela_list)

Transcript Type,Only Proximal ELA Transcript Count,TSS-overlapping ELA Transcript Count
str,u32,u32
"""lncRNA""",1139,819
"""retained_intron""",1032,466
"""protein_coding""",884,567
"""protein_coding_CDS_not_defined""",420,221
"""nonsense_mediated_decay""",239,154
"""processed_pseudogene""",116,40
"""TEC""",30,11
"""transcribed_processed_pseudoge…",21,5
"""transcribed_unprocessed_pseudo…",13,6


&nbsp;

&nbsp;

## Example: ENST00000778743.1

- ELA のみが周辺にある lncRNA として注目

In [7]:
HNscore_all = pl.read_parquet(
    "../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/annotation/HNscore_gencode_human_261001_HN2_transcript_all_gencode_annotation.parquet"
)

ENST00000778743 = HNscore_all.filter(
    pl.col("transcript_id") == "ENST00000778743.1"
)

display(ENST00000778743)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32
"""ENST00000778743.1""",306,17,83,289,207,2,42,205,null,null,"""lncRNA""","""ENSG00000301409.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""+""",4,"""URS0002A31C49""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## ENCODE-rE2G Result

- 仮説: lncRNA の近傍にある fanta.bio で定義された CRE が、別の遺伝子のエンハンサーとして働いている可能性を探索するために、解析

&nbsp;

&nbsp;

## 1. Metadata Process

- 使用したデータセットの組み合わせとアクセッションを表にする

In [8]:
encode_rE2G_metadata = pl.read_csv(
    "../../out/ENCODE_rE2G/config/config_biosamples_hs_atacseq_hic_list.tsv",
    separator="\t"
).drop(
    "DHS",
    "default_accessibility_feature",
    "HiC_resolution",
    "alt_TSS",
    "alt_genes",
    "HiC_type"
).with_columns(
    # Keep only SRR accessions from the absolute paths, joined with "; "
    pl.col("ATAC").str.extract_all(r"SRR\d+").list.join("; ").alias("ATAC"),
    pl.col("H3K27ac").str.extract_all(r"SRR\d+").list.join("; ").alias("H3K27ac"),
    # tissues.hic is the ENCODE tissue Megamap (ENCSR492BKE); otherwise keep each SRR once
    pl.when(pl.col("HiC_file").str.contains("tissues.hic"))
    .then(pl.lit("tissue Megamap (ENCSR492BKE)"))
    .otherwise(
        pl.col("HiC_file").str.extract_all(r"SRR\d+").list.unique(maintain_order=True).list.join("; ")
    )
    .alias("HiC_file")
).rename(
    {
        "ATAC": "HS-related ATAC-seq",
        "H3K27ac": "HS-related ChIP-seq (H3K27ac)",
        "HiC_file": "HS-related Hi-C"
    }
)

pl.Config.set_fmt_str_lengths(500)
display(encode_rE2G_metadata)

biosample,HS-related ATAC-seq,HS-related ChIP-seq (H3K27ac),HS-related Hi-C
str,str,str,str
"""GSE105028_hESC_42C_60min_atacseq_hic_chromap_chipseq""","""SRR6206775; SRR6206776""","""SRR6177968; SRR6177970""","""SRR6177928; SRR6177929"""
"""GSE280307_THP1_42C_120min_atacseq""","""SRR31114891; SRR31114887""",null,"""tissue Megamap (ENCSR492BKE)"""
"""GSE101731_HFF_44C_60min_atacseq""","""SRR5856839; SRR5856840""",null,"""tissue Megamap (ENCSR492BKE)"""
"""GSE101731_HFF_44C_120min_atacseq""","""SRR5856841; SRR5856842""",null,"""tissue Megamap (ENCSR492BKE)"""
"""GSE154744_K562_42C_12_5min_atacseq""","""SRR12267673; SRR12267674""",null,"""tissue Megamap (ENCSR492BKE)"""
"""GSE154744_K562_42C_25min_atacseq""","""SRR12267675; SRR12267676""",null,"""tissue Megamap (ENCSR492BKE)"""
"""GSE154744_K562_42C_40min_atacseq""","""SRR12267677; SRR12267678""",null,"""tissue Megamap (ENCSR492BKE)"""
"""GSE154744_K562_42C_30min_atacseq""","""SRR12267689; SRR12267690""",null,"""tissue Megamap (ENCSR492BKE)"""
"""GSE154744_K562_42C_60min_atacseq""","""SRR12267691; SRR12267692""",null,"""tissue Megamap (ENCSR492BKE)"""


&nbsp;

&nbsp;

&nbsp;

## Metadata (for figshare)

- 既に収集しているメタデータのうち、実際に使用したデータのみをフィルタリングする

In [9]:
# 左に寄せたいカラムリスト
# Columns to move to the left, in this order
LEFT_COLUMNS = ["sample_heat_id", "sample_control_id", "sample_name", "cell_type", "stress treatment temperature","stress treatment time", "control treatment temperature","reference_doi"]

&nbsp;

### 1. HS-related ATAC-seq

In [10]:
hs_related_atacseq_metadata = pl.read_csv(
    "../../Data/metadata/metadata_2610/atacseq_metadata_all_261001.tsv",
    separator="\t"
)


# "HS-related ATAC-seq" is "; "-joined SRR ids.
# run_accession is "heat SRR vs control SRR".
# Keep a metadata row when the heat-side SRR was used in ENCODE-rE2G.
used_hs_atac_srr = (
    encode_rE2G_metadata
    .select(
        pl.col("HS-related ATAC-seq")
        .str.extract_all(r"SRR\d+")
        .explode()
        .alias("srr")
    )
    .unique()
)

hs_related_atacseq_metadata_used = (
    hs_related_atacseq_metadata
    .with_columns(
        pl.col("run_accession").str.split(" vs ").list.first().str.strip_chars().alias("heat_srr")
    )
    .join(
        used_hs_atac_srr,
        left_on="heat_srr",
        right_on="srr",
        how="semi",
        maintain_order="left",
    )
    .drop("heat_srr")
).select(
    *LEFT_COLUMNS, pl.exclude(LEFT_COLUMNS) # 左に寄せたいカラム
).pipe(
    # Drop columns that are null in every remaining row
    lambda df: df.select(
        s.name for s in df if s.null_count() != df.height
    )
)

print(
    f"HS-related ATAC-seq SRR: {used_hs_atac_srr.height}; "
    f"metadata rows kept: {hs_related_atacseq_metadata_used.height} / {hs_related_atacseq_metadata.height}"
)

# for figshare
hs_related_atacseq_metadata_used.write_csv(
    "../../out/figshare/encode_re2g/metadata/metadata_hs_related_atacseq4encode_re2g.tsv",
    separator="\t"
)

pl.Config.set_tbl_rows(2)
display(hs_related_atacseq_metadata_used)

HS-related ATAC-seq SRR: 28; metadata rows kept: 32 / 34


sample_heat_id,sample_control_id,sample_name,cell_type,stress treatment temperature,stress treatment time,control treatment temperature,reference_doi,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_series_id,cell_line,genome_build,sample_accession,is_chip_atlas_registered,library_selection,secondary_study_accession,study_title,description,library_strategy,library_source,library_layout,library_construction_protocol,experiment_alias,center_name,scientific_name,tax_id,study_accession,secondary_sample_accession,treatment,stress_with,replicate,sample_characteristics_ch1,preconditioning,genotype
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str
"""SRX3316069""","""SRX3316067""","""hs_fast_atac-seq_rep1 vs fast_atac-seq_rep1""","""H9 human Embryonic Stem Cell Line""","""42 C""","""60 min.""","""37 C (inferred)""","""https://doi.org/10.1016/j.molcel.2018.07.012""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6206775 vs SRR6206773""","""HS Fast ATAC-Seq Rep1 vs Fast ATAC-Seq Rep1""","""GSM2829008 vs GSM2829006""","""Public on Aug 03 2018""","""Oct 24 2017""","""May 15 2019""","""SRA""","""H9 hESCs""","""Homo sapiens""",9606,"""genomic DNA""","""fast ATAC-Seq, Fast ATAC-Seq under heat shock conditions, HS_ATAC-Seq.bw vs fast ATAC-Seq, Fast ATAC-Seq under normal conditions, ATAC-Seq.bw""","""GPL11154""","""Illumina HiSeq 2000""","""other""","""genomic""","""ATAC-seq""","""GSM2829008 vs GSM2829006""","""GSE105028""","""hESC""","""hg38""","""SAMN07828432 vs SAMN07828392""",false,"""other""","""SRP120027""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""Illumina HiSeq 2000 sequencing: GSM2829008: HS Fast ATAC-Seq Rep1 Homo sapiens ATAC-seq vs Illumina HiSeq 2000 sequencing: GSM2829006: Fast ATAC-Seq Rep1 Homo sapiens ATAC-seq""","""ATAC-seq""","""GENOMIC""","""PAIRED""","""For Hi-C, cells were crosslinked with 1% formaldehyde for 10 min at room temperature and quenched with glycine, nuclei were isolated and chromatin digested by HindIII, filled in with biotin-dCTP, and ligated. After ligation, nuclei were digested by ProteinaseK and chromatin were reverse-crosslinked overnight, followed by ethanol precipitation to get purified DNA. DNA were sonicated to 200-500 bp fragment and enriched with streptavidin dynal beads, followed by illumina library preparation. For Ch…","""GSM2829008 vs GSM2829006""","""GEO""","""Homo sapiens""",9606,"""PRJNA414442""","""SRS2621656 vs SRS2621654""",null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRX10705951""","""SRX10705949""","""atacseq_lymphblastoid_cells_t21_42c_rep2_technical2 vs atacseq_lymphblastoid_cells_t21_37c_rep2_technical2""","""lymphoblastoid""","""42 C""","""60 min.""","""37 C (inferred)""","""https://doi.org/10.1371/journal.pone.0307375""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR14352104 vs SRR14352102""","""ATACseq_Lymphblastoid cells_T21_42C_rep2_technical2 vs ATACseq_Lymphblastoid cells_T21_37C_rep2_technical2""","""GSM5269436 vs GSM5269434""","""Public on Apr 21 2023""","""Apr 28 2021""","""Apr 21 2023""","""SRA""","""lymphoblastoid-EBV immortalized""","""Homo sapiens""",9606,"""genomic DNA""","""Ethan_T21, genomic DNA (ATAC-seq), Ethan42merged_peaks_clean.narrowPeak vs Ethan_T21, genomic DNA (ATAC-seq), Ethan37merged_peaks_clean.narrowPeak""","""GPL18573""","""Illumina

&nbsp;

### 2. HS-related ChIP-seq

- データが少ないので manual で対応

In [11]:
hs_related_chipseq_metadata = pl.read_csv(
    "../../Data/metadata/metadata_2610/chipseq_histone_modification_metadata_all_261001.tsv",
    separator="\t"
).filter(
    pl.col("run_accession").str.contains("SRR6177968") | pl.col("run_accession").str.contains("SRR6177970")
).with_columns(
    pl.lit("37 C (inferred)").alias("control treatment temperature") # このデータのカラムを追加し忘れていた
).with_columns(
    pl.lit("H9 human Embryonic Stem Cell Line").alias("cell_type") # 今回フィルタリングしたデータは、1つのプロジェクト由来なので、まとめて cell type を指定
).select(
    *LEFT_COLUMNS, pl.exclude(LEFT_COLUMNS) # 左に寄せたいカラム
).drop(
    "fastq_galaxy", "fastq_ftp", "fastq_aspera", "fastq_md5", "run_accession_right", "tax_lineage"
).pipe(
    # Drop columns that are null in every remaining row
    lambda df: df.select(
        s.name for s in df if s.null_count() != df.height
    )
)

# for figshare
hs_related_chipseq_metadata.write_csv(
    "../../out/figshare/encode_re2g/metadata/metadata_hs_related_chipseq4encode_re2g.tsv",
    separator="\t"
)

pl.Config.set_tbl_cols(400)
display(hs_related_chipseq_metadata)

sample_heat_id,sample_control_id,sample_name,cell_type,stress treatment temperature,stress treatment time,control treatment temperature,reference_doi,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,genome_build,biosample,sra,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,base_count,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias,sample_contact_state,condition,sample_description_right,library_strategy_right,center_name
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str
"""SRX3288592""","""SRX3288591""","""hs_h3k27ac_chip-seq_rep vs h3k27ac_chip-seq_rep""","""H9 human Embryonic Stem Cell Line""","""42 C""","""60 min.""","""37 C (inferred)""","""https://doi.org/10.1016/j.molcel.2018.07.012""","""ChIP""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6177970 vs SRR6177969""","""HS H3K27ac ChIP-Seq Rep vs H3K27ac ChIP-Seq Rep""","""GSM2816651 vs GSM2816650""","""Public on Aug 03 2018""","""Oct 16 2017""","""May 15 2019""","""SRA""",1,"""H9 hESCs""","""Homo sapiens""",9606,"""genomic DNA""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""","""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""361102""","""China""","""Illumina HiSeq 2000""","""ChIP""","""genomic""","""ChIP-Seq""","""GSM2816651 vs GSM2816650""","""NONE""","""GSE105028""",0,"""SRX3288592 vs SRX3288591""","""H9 human Embryonic Stem Cell Line""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN07790053 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN07790054""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3288592 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX3288591""","""Illumina HiSeq 2000 sequencing: GSM2816651: HS H3K27ac ChIP-Seq Rep Homo sapiens ChIP-Seq vs Illumina HiSeq 2000 sequencing: GSM2816650: H3K27ac ChIP-Seq Rep Homo sapiens ChIP-Seq""","""SAMN07790053 vs SAMN07790054""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""SRA619794""","""ChIP""","""2018-08-04""","""252326053;260678980 vs 305381145;310819618""","""ILLUMINA""","""Generic""","""ChIP-Seq under heat shock conditions, H3K27ac_HS.bw vs ChIP-Seq under normal conditions, H3K27ac_NT.bw""","""SRP120027""","""8548663 vs 10283229""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""GENERATED_FILE;GENERATED_FILE""","""HS H3K27ac ChIP-Seq Rep vs H3K27ac ChIP-Seq Rep""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM2816651: HS H3K27ac ChIP-Seq Rep Homo sapiens ChIP-Seq vs Illumina HiSeq 2000 sequencing: GSM2816650: H3K27ac ChIP-

&nbsp;

### 3. HS-related Hi-C data

- アクセッション番号が少ないので、マニュアルで対応する
- Hi-ChIP data が含まれていることに注意
- これは、`sample_heat_id` が run accession になっていたので、こちらを標的にする

In [12]:
hs_related_hic_metadata = pl.read_csv(
    "../../Data/metadata/metadata_2610/hic_metadata_all_261001.tsv",
    separator="\t"
).filter(
    (pl.col("sample_heat_id").str.contains("SRR6177928"))
    | (pl.col("sample_heat_id").str.contains("SRR6177929"))
    | (pl.col("sample_heat_id").str.contains("SRR9019504"))
    | (pl.col("sample_heat_id").str.contains("SRR9019505"))
).pipe(
    # Drop columns that are null in every remaining row
    lambda df: df.select(
        s.name for s in df if s.null_count() != df.height
    )
).with_columns(
    # cell type カラムがなかったため、マニュアルで対応
    pl.when(
        pl.col("sample_source_name_ch1") == "H9 hESCs"
    ).then(
        pl.lit("H9 human Embryonic Stem Cell Line")
    ).when(
        pl.col("sample_source_name_ch1") == "K562 immortalized cell line"
    ).then(
        pl.lit("K562")
    ).otherwise(
        pl.lit("Unknown")
    ).alias("cell_type")
).select(
    *LEFT_COLUMNS, pl.exclude(LEFT_COLUMNS) # 左に寄せたいカラム
)

# for figshare
hs_related_hic_metadata.write_csv(
    "../../out/figshare/encode_re2g/metadata/metadata_hs_related_hic4encode_re2g.tsv",
    separator="\t"
)

pl.Config.set_tbl_rows(4)
display(hs_related_hic_metadata)

sample_heat_id,sample_control_id,sample_name,cell_type,stress treatment temperature,stress treatment time,control treatment temperature,reference_doi,protocol,organism,read_type,data_source,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,condition,library_strategy,genome_build,biosample,sra,sample_contact_laboratory,sample_contact_department
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str
"""SRR6177928""","""SRR6177926""","""hs_hic_rep1 vs hic_rep1""","""H9 human Embryonic Stem Cell Line""","""42 C""","""60 min.""","""37 C (inferred)""","""https://doi.org/10.1016/j.molcel.2018.07.012""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRX3288550 vs SRX3288548""","""HS HiC _Rep1 vs HiC _Rep1""","""GSM2816609 vs GSM2816607""","""Public on Aug 03 2018""","""Oct 16 2017""","""May 15 2019""","""SRA""",1,"""H9 hESCs""","""Homo sapiens""",9606,"""genomic DNA""","""in-situ Hi-C under heat shock conditions, HS_H9_Hi-C.1kb.intra.hic vs in-situ Hi-C under normal conditions, H9_Hi-C.1kb.intra.hic""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""","""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""FUJIAN""",361102,"""China""","""Illumina HiSeq 2000""","""other""","""genomic""","""OTHER""","""GSM2816609 vs GSM2816607""","""NONE""","""GSE105028""",0,"""SRX3288550 vs SRX3288548""","""H9 human Embryonic Stem Cell Line""","""under heat shock conditions vs under normal conditions""","""HiC-Seq""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN07790035 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN07790044""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3288550 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX3288548""",null,null
"""SRR6177929""","""SRR6177927""","""hs_hic_rep2 vs hic_rep2""","""H9 human Embryonic Stem Cell Line""","""42 C""","""60 min.""","""37 C (inferred)""","""https://doi.org/10.1016/j.molcel.2018.07.012""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRX3288551 vs SRX3288549""","""HS HiC _Rep2 vs HiC _Rep2""","""GSM2816610 vs GSM2816608""","""Public on Aug 03 2018""","""Oct 16 2017""","""May 15 2019""","""SRA""",1,"""H9 hESCs""","""Homo sapiens""",9606,"""genomic DNA""","""in-situ Hi-C under heat shock conditions, HS_H9_Hi-C.1kb.intra.hic vs in-situ Hi-C under normal conditions, H9_Hi-C.1kb.intra.hic""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""","""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""FUJIAN""",361102,"""China""","""Illumina HiSeq 2000""","""other""","""genomic""","""OTHER""","""GSM2816610 vs GSM2816608""","""NONE""","""GSE105028""",0,"""SRX3288551 vs SRX3288549""","""H9 human Embryonic Stem Cell Line""","""under heat shock conditions vs under normal conditions""","""HiC-Seq""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN07790042 vs https://www.ncbi.nlm.nih.gov/biosample/SAMN07790043""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3288551 vs https://www.ncbi.nlm.nih.gov/sra?term=SRX3288549""",null,null
"""SRR9019504""","""SRR9019506""","""biosample_8606_4132_58665_hyl2cbgx2_k562_insituhic_hs_1_cgatgt vs biosample_8606_4132_58664_hyl2cbgx2_k562_insituhic_nhs_1_atcacg""","""K562""","

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. ENCODE-rE2G Result (heat-stress)

- NOTE: `isSelfPromoter` を False の状態でカウントした 相互作用を掲載する！

In [13]:
encode_re2g_result = pl.read_parquet(
    "../../out/ENCODE_rE2G/parquet_file/encode_re2g_hs_join_v261113.parquet"
).filter(
    pl.col("isSelfPromoter") == False
)

print(f"encode_re2g_result count: {encode_re2g_result.select(["cre_id", "target_cre_id"]).n_unique()}")
pl.Config.set_tbl_rows(10)
display(encode_re2g_result)

encode_re2g_result count: 368398


ENCODE-rE2G.Score,re2g_dataset,cre_id,cre_name,cre_chr,fantabio_cre_class,enhancer_cre_overlap_bp,enhancer_cre_overlap_frac_of_fantabio_cre,TargetGene,target_cre_id,target_cre_name,target_fantabio_cre_class,class,isSelfPromoter,distanceToTSS,re2g_enhancer_locus,fantabio_enhancer_cre,re2g_features,target_tss_info,target_cre_info,target_cre_overlap_info
f64,str,str,str,str,str,i64,f64,str,str,str,str,str,bool,f64,struct[3],struct[3],struct[8],struct[5],struct[10],struct[2]
0.194134,"""GSE267931_HUVEC_39C_120min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",358,0.854415,"""TRIM62""","""FCHS_10021""","""cp3@TRIM62""","""PLA""","""genic""",false,84675.0,"{""chr1"",33097145,33097645}","{33097084,33097503,0.614679}","{33182070,""ENSG00000116525"",0.524194,null,0.013442,13,1,506.041996}","{""chr1"",33181820,33182320,0,""-""}","{""chr1"",33181859,33182582,0,""."",33182047,33182188,""0,0,204"",""directionality:-0.999247|class:PLA"",-0.999247}","{461,0.922}"
0.342709,"""GSE154744_K562_42C_25min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",353,0.842482,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15223.0,"{""chr1"",33097150,33097650}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.383964,null,0.044833,2,1,220.597881}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.320048,"""GSE154744_K562_42C_60min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",288,0.687351,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15288.0,"{""chr1"",33097215,33097715}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.402521,null,0.044762,4,1,253.0904}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.312414,"""GSE267931_HUVEC_39C_120min""","""FCHS_10001""","""FCHS_10001""","""chr1""","""ELA""",358,0.854415,"""AZIN2""","""FCHS_9997""","""cp4@AZIN2""","""ELA""","""genic""",false,15218.0,"{""chr1"",33097145,33097645}","{33097084,33097503,0.614679}","{33082177,""ENSG00000142920"",0.466515,null,0.044839,4,1,506.041996}","{""chr1"",33081927,33082427,0,""+""}","{""chr1"",33081878,33082317,0,""."",33081878,33082179,""255,202,74"",""directionality:1|class:ELA"",1.0}","{390,0.78}"
0.291194,"""GSE173536_lymphblastoid_d21_42C_60min""","""FCHS_100026""","""FCHS_100026""","""chr12""","""ELA""",201,1.0,"""RASSF9""","""FCHS_100044""","""cp1@RASSF9""","""PLA""","""intergenic""",false,411922.0,"{""chr12"",85423911,85425325}","{85423957,85424158,0.5}","{85836540,""ENSG00000198774"",0.357834,null,0.00446,9,1,0.0}","{""chr12"",85836290,85836790,0,""-""}","{""chr12"",85836188,85836732,0,""."",85836213,85836732,""0,0,255"",""directionality:-1|class:PLA"",-1.0}","{442,0.884}"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.21537,"""GSE101731_HFF_44C_60min""","""FCHS_99979""","""FCHS_99979""","""chr12""","""ELA""",396,0.938389,"""SLC6A15""","""FCHS_99974""","""cp9@SLC6A15""","""PLA""","""intergenic""",false,10163.0,"{""chr12"",84922742,84923242}","{84922716,84923138,-0.948148}","{84912829,""ENSG00000072041"",0.581099,null,0.077244,1,1,0.0}","{""chr12"",84912579,84913079,0,""-""}","{""chr12"",84912565,84913337,0,""."",84912757,84912903,""0,0,204"",""directionality:-0.99533|class:PLA"",-0.99533}","{500,1.0}"
0.311435,"""GSE101731_HFF_44C_120min""","""FCHS_99987""","""cp1@TSPAN19""","""chr12""","""PLA""",423,1.0,"""SLC6A15""","""FCHS_99974""","""cp9@SLC6A15""","""PLA""","""genic""",false,103772.0,"{""chr12"",85016351,85016851}","{85016374,85016797,-0.977465}","{84912829,""ENSG00000072041"",0.486403,null,0.014404,3,1,0.0}","{""chr12"",84912579,84913079,0,""-""}","{""chr12"",84912565,84913337,0,""."",84912757,84912903,""0,0,204"",""directionality:-0.99533|class:PLA"",-0.99533}","{500,1.0}"
0.291244,"""GSE101731_HFF_44C_60min""","""FCHS_99987""","""cp1@TSPAN19"

&nbsp;

### Convert to TSV (for figshare)

In [14]:
left_columns = [
    "re2g_dataset",
    "cre_id",
    "cre_name",
    "target_cre_id",
    "ENCODE-rE2G.Score",
    "target_cre_name",
    "fantabio_cre_class",
    "target_fantabio_cre_class"
]


encode_re2g_result_flat = (
    encode_re2g_result
).select(
    *left_columns, pl.exclude(left_columns) # 左に寄せたいカラム
).drop(
    "class",
    "isSelfPromoter",
    "distanceToTSS",
    "re2g_enhancer_locus",
    "fantabio_enhancer_cre",
    "re2g_features",
    "target_tss_info",
    "target_cre_info",
    "target_cre_overlap_info",
    "TargetGene",
    "cre_chr",
    "enhancer_cre_overlap_bp",
    "enhancer_cre_overlap_frac_of_fantabio_cre",
    "cre_name",
    "target_cre_name"
).rename(
    {
        "cre_id": "enhancer_cre_id",
        # "cre_name": "enhancer_cre_name",
        "target_cre_id": "target_cre_id",
        # "target_cre_name": "target_cre_name",
        "fantabio_cre_class": "enhancer_fantabio_cre_class",
        "target_fantabio_cre_class": "target_fantabio_cre_class"
    }
).sort(
   by=["enhancer_cre_id", "target_cre_id", "ENCODE-rE2G.Score"],
   descending=[False, False, True]
)

print(f"encode-rE2G result count: {encode_re2g_result_flat.select(['enhancer_cre_id', 'target_cre_id']).n_unique()}")
display(encode_re2g_result_flat)

encode-rE2G result count: 368398


re2g_dataset,enhancer_cre_id,target_cre_id,ENCODE-rE2G.Score,enhancer_fantabio_cre_class,target_fantabio_cre_class
str,str,str,f64,str,str
"""GSE267931_HUVEC_39C_120min""","""FCHS_10001""","""FCHS_10021""",0.194134,"""ELA""","""PLA"""
"""GSE154744_K562_42C_25min""","""FCHS_10001""","""FCHS_9997""",0.342709,"""ELA""","""ELA"""
"""GSE154744_K562_42C_60min""","""FCHS_10001""","""FCHS_9997""",0.320048,"""ELA""","""ELA"""
"""GSE267931_HUVEC_39C_120min""","""FCHS_10001""","""FCHS_9997""",0.312414,"""ELA""","""ELA"""
"""GSE173536_lymphblastoid_d21_42C_60min""","""FCHS_100026""","""FCHS_100044""",0.291194,"""ELA""","""PLA"""
…,…,…,…,…,…
"""GSE101731_HFF_44C_60min""","""FCHS_99979""","""FCHS_99974""",0.21537,"""ELA""","""PLA"""
"""GSE101731_HFF_44C_120min""","""FCHS_99987""","""FCHS_99974""",0.311435,"""PLA""","""PLA"""
"""GSE101731_HFF_44C_60min""","""FCHS_99987""","""FCHS_99974""",0.291244,"""PLA""","""PLA"""


In [15]:
# for figshare
encode_re2g_result_flat.write_csv(
    "../../out/figshare/encode_re2g/encode_re2g_result_hs_related_dataset_all.tsv",
    separator="\t"
)

&nbsp;

&nbsp;

&nbsp;

## 2-1. Unique Interaction Count

In [16]:
e2g_join_unique_count = encode_re2g_result.group_by(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "target_cre_id",
    "target_cre_name",
    "target_fantabio_cre_class"
).agg(
    pl.col("re2g_dataset").unique().alias("re2g_dataset"),
    pl.col("re2g_dataset").n_unique().alias("re2g_dataset_count"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").alias("enhancer_cre_overlap_frac_of_fantabio_cre"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").min().alias("enhancer_cre_overlap_frac_of_fantabio_cre_min"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").max().alias("enhancer_cre_overlap_frac_of_fantabio_cre_max"),
    pl.col("enhancer_cre_overlap_frac_of_fantabio_cre").mean().alias("enhancer_cre_overlap_frac_of_fantabio_cre_mean"), # 該当するデータセットで同定された enhancer と fantabio の CRE の重複率の平均値

    pl.col("ENCODE-rE2G.Score").alias("ENCODE-rE2G.Score"),
    pl.col("ENCODE-rE2G.Score").min().alias("ENCODE-rE2G.Score_min"),
    pl.col("ENCODE-rE2G.Score").max().alias("ENCODE-rE2G.Score_max"),
    pl.col("ENCODE-rE2G.Score").mean().alias("ENCODE-rE2G.Score_mean"), # 該当するデータセットで同定された enhancer と target promoter の相互作用のスコアの平均値

    # pl.col("ENCODE-rE2G.Score").quantile(0.25).alias("ENCODE-rE2G.Score_q1"),
    # pl.col("ENCODE-rE2G.Score").median().alias("ENCODE-rE2G.Score_median"),
    # pl.col("ENCODE-rE2G.Score").quantile(0.75).alias("ENCODE-rE2G.Score_q3"),
    # pl.col("ENCODE-rE2G.Score").std().alias("ENCODE-rE2G.Score_std"),
).sort(
    by=["re2g_dataset_count", "ENCODE-rE2G.Score_mean"],
    descending=True
)

print(f"ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): {e2g_join_unique_count.select(['cre_id', 'target_cre_id']).n_unique()}")
print(f"ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level) (all dataset): {e2g_join_unique_count.filter(pl.col("re2g_dataset_count") == e2g_join_unique_count["re2g_dataset_count"].max()).select(['cre_id', 'target_cre_id']).n_unique()}")
display(e2g_join_unique_count)

ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): 368398
ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level) (all dataset): 977


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64
"""FCHS_272143""","""FCHS_272143""","""PLA""","""FCHS_272148""","""cp1@MTFP1""","""PLA""","[""GSE173536_lymphblastoid_t21_42C_60min"", ""GSE173536_lymphblastoid_d21_42C_60min"", … ""GSE154744_K562_42C_30min_intact_hic""]",14,"[0.752244, 0.748654, … 0.906643]",0.560144,0.906643,0.767248,"[0.995691, 0.995606, … 0.971064]",0.971064,0.995691,0.988938
"""FCHS_272142""","""FCHS_272142""","""PLA""","""FCHS_272148""","""cp1@MTFP1""","""PLA""","[""GSE173536_lymphblastoid_t21_42C_60min"", ""GSE173536_lymphblastoid_d21_42C_60min"", … ""GSE154744_K562_42C_30min_intact_hic""]",14,"[1.0, 1.0, … 1.0]",0.703704,1.0,0.961698,"[0.995691, 0.995606, … 0.971064]",0.971064,0.995691,0.988938
"""FCHS_208523""","""cp2@PLAUR""","""PLA""","""FCHS_208525""","""cp5@PLAUR""","""PLA""","[""GSE154744_K562_42C_60min"", ""GSE173536_lymphblastoid_d21_42C_60min"", … ""GSE154744_K562_42C_30min_intact_hic""]",14,"[0.900362, 0.727355, … 0.930254]",0.727355,0.946558,0.849249,"[0.995005, 0.994978, … 0.941883]",0.941883,0.995005,0.988152
"""FCHS_168111""","""cp6@NCOR1""","""PLA""","""FCHS_168114""","""cp4@PIGL""","""PLA""","[""GSE154744_K562_42C_40min"", ""GSE154744_K562_42C_60min"", … ""GSE105028_hESC_42C_60min""]",14,"[0.775947, 1.0, … 1.0]",0.775947,1.0,0.944928,"[0.99309, 0.992987, … 0.960316]",0.960316,0.99309,0.98727
"""FCHS_133635""","""FCHS_133635""","""PLA""","""FCHS_133639""","""cp3@OCA2""","""PLA""","[""GSE101731_HFF_44C_60min"", ""GSE209675_MCF7_42C_60min"", … ""GSE154744_K562_42C_30min_intact_hic""]",14,"[0.889186, 0.933244, … 0.667557]",0.660881,0.933244,0.79506,"[0.996407, 0.992461, … 0.953398]",0.953398,0.996407,0.986245
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_580787""","""FCHS_580787""","""ELA""","""FCHS_406577""","""cp1@PSKH2""","""ELA""","[""GSE209675_MCF7_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.179003],0.179003,0.179003,0.179003
"""FCHS_17957""","""FCHS_17957""","""ELA""","""FCHS_18071""","""cp1@CCN1,cp6@CCNA2""","""PLA""","[""GSE173536_lymphblastoid_t21_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.179002],0.179002,0.179002,0.179002
"""FCHS_17956""","""FCHS_17956""","""ELA""","""FCHS_18071""","""cp1@CCN1,cp6@CCNA2""","""PLA""","[""GSE173536_lymphblastoid_t21_42C_60min""]",1,[1.0],1.0,1.0,1.0,[0.179002],0.179002,0.179002,0.179002


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Target Promoter proximal to up5p transcripts

- 標的のプロモーターの近傍にある 転写産物のうち、up5p transcript lists が含まれる CRE-CRE pairs をカウントする

In [17]:
# target promoter side transcript candidate
fantabio_HN2_up5p_target_transcript_candidate = up5p_transcript_cre_pairs.select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class",
    "transcript_id",
    "HN2",
    "transcript_type",
    "gene_id",
    "gene_symbol",
    "strand",
    "RNACentral_ID",
    "annotation_remarks",
    "tags"
).unique().rename(
    # assign frag name "t" to target promoter side
    {
        "cre_id": "target_cre_id",
        "cre_name": "target_cre_name",
        "fantabio_cre_class": "target_fantabio_cre_class",
        "transcript_id": "t_transcript_id",
        "HN2": "t_HN2",
        "transcript_type": "t_transcript_type",
        "gene_id": "t_gene_id",
        "gene_symbol": "t_gene_symbol",
        "strand": "t_strand",
        "RNACentral_ID": "t_RNACentral_ID",
        "annotation_remarks": "t_annotation_remarks",
        "tags": "t_tags"
    }
)

display(fantabio_HN2_up5p_target_transcript_candidate)

target_cre_id,target_cre_name,target_fantabio_cre_class,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_72808""","""cp6@STIP1""","""ELA""","""ENST00000688681.2""",127,"""lncRNA""","""ENSG00000288852.2""",null,"""-""","""URS0002A2E719""","[""Assembled from PacBio CLS3 reads"", ""CLS3 project""]","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]"
"""FCHS_164275""","""cp20@MYO1C""","""ELA""","""ENST00000575011.1""",91,"""nonsense_mediated_decay""","""ENSG00000197879.18""","""MYO1C""","""-""",null,null,"[""mRNA_start_NF"", ""cds_start_NF""]"
"""FCHS_355066""","""cp8@RGL2""","""ELA""","""ENST00000908155.1""",132,"""protein_coding""","""ENSG00000237441.11""","""RGL2""","""-""",null,"[""DanaFarber""]","[""basic"", ""TAGENE""]"
"""FCHS_18812""","""cp2@GLMN""","""PLA""","""ENST00000896610.1""",58,"""protein_coding""","""ENSG00000174842.18""","""GLMN""","""-""",null,"[""DanaFarber""]","[""basic"", ""TAGENE""]"
"""FCHS_190464""","""cp4@ZNF24""","""PLA""","""ENST00000931765.1""",66,"""protein_coding""","""ENSG00000172466.17""","""ZNF24""","""-""",null,"[""LRGASP""]","[""basic"", ""TAGENE"", … ""CCDS""]"
…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_195346""","""cp1@GZMM""","""PLA""","""ENST00000264553.6""",81,"""protein_coding""","""ENSG00000197540.8""","""GZMM""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
"""FCHS_151844""","""cp5@BMERB1""","""ELA""","""ENST00000561813.5""",58,"""nonsense_mediated_decay""","""ENSG00000166780.12""","""BMERB1""","""+""",null,null,[]
"""FCHS_371580""","""cp1@MRPL18""","""PLA""","""ENST00000480842.1""",206,"""protein_coding_CDS_not_defined""","""ENSG00000112110.11""","""MRPL18""","""+""",null,"[""mitochondrial ribosomal protein L18""]",[]


In [18]:
e2g_join_unique_count_join = e2g_join_unique_count.join(
    fantabio_HN2_up5p_target_transcript_candidate,
    on=["target_cre_id", "target_cre_name", "target_fantabio_cre_class"],
    how="inner"
).sort(
    by=["t_HN2", "re2g_dataset_count"],
    descending=[True, True]
)

print(f"ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): {e2g_join_unique_count_join.select(['cre_id', 'target_cre_id']).n_unique()}")
print(f"ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level) (all dataset): {e2g_join_unique_count_join.filter(pl.col("re2g_dataset_count") == e2g_join_unique_count_join["re2g_dataset_count"].max()).select(['cre_id', 'target_cre_id']).n_unique()}")
print(f"ENCODE-rE2G enhancer - target region interaction with up5p transcripts (fanta.bio cre level): {e2g_join_unique_count_join.select(['t_transcript_id']).n_unique()}")
display(e2g_join_unique_count_join)

ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level): 110273
ENCODE-rE2G enhancer - target region interaction unique pair count (fanta.bio cre level) (all dataset): 487
ENCODE-rE2G enhancer - target region interaction with up5p transcripts (fanta.bio cre level): 13379


cre_id,cre_name,fantabio_cre_class,target_cre_id,target_cre_name,target_fantabio_cre_class,re2g_dataset,re2g_dataset_count,enhancer_cre_overlap_frac_of_fantabio_cre,enhancer_cre_overlap_frac_of_fantabio_cre_min,enhancer_cre_overlap_frac_of_fantabio_cre_max,enhancer_cre_overlap_frac_of_fantabio_cre_mean,ENCODE-rE2G.Score,ENCODE-rE2G.Score_min,ENCODE-rE2G.Score_max,ENCODE-rE2G.Score_mean,t_transcript_id,t_HN2,t_transcript_type,t_gene_id,t_gene_symbol,t_strand,t_RNACentral_ID,t_annotation_remarks,t_tags
str,str,str,str,str,str,list[str],u32,list[f64],f64,f64,f64,list[f64],f64,f64,f64,str,i64,str,str,str,str,str,list[str],list[str]
"""FCHS_28114""","""FCHS_28114""","""PLA""","""FCHS_28106""","""cp4@HSPA6""","""PLA""","[""GSE154744_K562_42C_12_5min"", ""GSE101731_HFF_44C_60min"", … ""GSE105028_hESC_42C_60min""]",14,"[0.925816, 0.934718, … 0.888724]",0.734421,1.0,0.86011,"[0.962253, 0.956158, … 0.317242]",0.317242,0.962253,0.85706,"""ENST00000309758.6""",335,"""protein_coding""","""ENSG00000173110.8""","""HSPA6""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
"""FCHS_28117""","""FCHS_28117""","""ELA""","""FCHS_28106""","""cp4@HSPA6""","""PLA""","[""GSE154744_K562_42C_30min"", ""GSE154744_K562_42C_40min"", … ""GSE280307_THP1_42C_120min""]",13,"[1.0, 1.0, … 1.0]",0.546939,1.0,0.905651,"[0.868326, 0.866603, … 0.362252]",0.362252,0.868326,0.694225,"""ENST00000309758.6""",335,"""protein_coding""","""ENSG00000173110.8""","""HSPA6""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
"""FCHS_28112""","""FCHS_28112""","""ELA""","""FCHS_28106""","""cp4@HSPA6""","""PLA""","[""GSE173536_lymphblastoid_t21_42C_60min"", ""GSE267931_HUVEC_39C_120min"", … ""GSE209675_MCF7_42C_60min""]",13,"[0.782114, 1.0, … 0.785366]",0.765854,1.0,0.867292,"[0.947753, 0.883595, … 0.281932]",0.281932,0.947753,0.722447,"""ENST00000309758.6""",335,"""protein_coding""","""ENSG00000173110.8""","""HSPA6""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
"""FCHS_28115""","""FCHS_28115""","""ELA""","""FCHS_28106""","""cp4@HSPA6""","""PLA""","[""GSE154744_K562_42C_25min"", ""GSE154744_K562_42C_30min"", … ""GSE280307_THP1_42C_120min""]",8,"[1.0, 1.0, … 1.0]",0.858065,1.0,0.982258,"[0.856925, 0.819678, … 0.307724]",0.307724,0.856925,0.701339,"""ENST00000309758.6""",335,"""protein_coding""","""ENSG00000173110.8""","""HSPA6""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
"""FCHS_28116""","""FCHS_28116""","""ELA""","""FCHS_28106""","""cp4@HSPA6""","""PLA""","[""GSE154744_K562_42C_30min"", ""GSE154744_K562_42C_60min"", … ""GSE280307_THP1_42C_120min""]",8,"[0.791232, 0.77453, … 0.920668]",0.77453,0.920668,0.856211,"[0.868326, 0.862714, … 0.362252]",0.362252,0.868326,0.632928,"""ENST00000309758.6""",335,"""protein_coding""","""ENSG00000173110.8""","""HSPA6""","""+""",null,"[""MANE_select""]","[""basic"", ""Ensembl_canonical"", … ""CCDS""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FCHS_384323""","""FCHS_384323""","""ELA""","""FCHS_384310""","""cp2@SEMA3C""","""PLA""","[""GSE101731_HFF_44C_120min""]",1,[0.527778],0.527778,0.527778,0.527778,[0.183831],0.183831,0.183831,0.183831,"""ENST00000878446.1""",44,"""protein_coding""","""ENSG00000075223.15""","""SEMA3C""","""-""",null,"[""DanaFarber""]","[""basic"", ""TAGENE"", … ""CCDS""]"
"""FCHS_511916""","""FCHS_511916""","""ELA""","""FCHS_198199""","""cp1@MIR7-3HG""","""PLA""","[""GSE280307_THP1_42C_120min""]",1,[1.0],1.0,1.0,1.0,[0.327206],0.327206,0.327206,0.327206,"""ENST00000317292.10""",44,"""lncRNA""","""ENSG00000176840.15""","""MIR7-3HG""","""+""","""URS0002A2D70C""","[""Assembled from PacBio CLS reads - CAGE_PolyA_HiSeq (GSE93848)"", ""Assembled from PacBio CLS3 reads"", ""CLS3 project""]","[""TAGENE""]"
"""FCHS_429562""","""FCHS_429562""","""ELA""","""FCHS_429605""","""cp4@ST6GALNAC6""","""PLA""","[""GSE267931_HUVEC_39C_120min""]",1,[1.0],1.0,1.0,1.0,[0.327098],0.327098,0.327098,0.32